# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets and for each, the fields and columns defined in the schema, referencing their `@id` for consistency.

In [ ]:
print("Available record sets with their field and column IDs:")
for recset in metadata.record_sets:
    print(f"- Record Set @id: {recset.id}, Name: {getattr(recset, 'name', 'N/A')}")
    for field in recset.fields:
        print(f"    - Field @id: {field.id}, Name: {getattr(field, 'name', 'N/A')}")
        if hasattr(field, "columns"):
            for column in field.columns:
                print(f"        - Column @id: {column.id}, Name: {getattr(column, 'name', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Replace `<record_set_id>` with an actual record set `@id` when known (from above cell output).

In [ ]:
# List all record set @ids
record_sets = [recset.id for recset in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for {record_set_id}:", dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head(2))
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize numeric fields, and group categorical data.

**You can adjust field IDs as needed based on the data overview above.**

In [ ]:
# Example EDA: Pick the first available record set with a numeric field
import numpy as np

chosen_record_set = None
numeric_field_id = None
group_field_id = None

for recset in metadata.record_sets:
    if recset.id in dataframes and len(dataframes[recset.id]):
        df = dataframes[recset.id]
        # Find a likely numeric field
        for field in recset.fields:
            if hasattr(field, 'data_type') and field.data_type in ['Float', 'Number', 'Integer']:
                if field.id in df.columns:
                    chosen_record_set = recset.id
                    numeric_field_id = field.id
                    break
        # Find a group field (categorical string field)
        for field in recset.fields:
            if hasattr(field, 'data_type') and field.data_type in ['Text']:
                if field.id in df.columns:
                    group_field_id = field.id
                    break
        if chosen_record_set:
            break

if chosen_record_set and numeric_field_id:
    print(f"Using record set '{chosen_record_set}' and numeric field '{numeric_field_id}'.")
    df = dataframes[chosen_record_set]
    # Convert the field to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Example threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Optionally group by a categorical field if found
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable record set with a numeric field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here's an example using matplotlib to plot the normalized numeric field distribution.

In [ ]:
import matplotlib.pyplot as plt

if chosen_record_set and numeric_field_id and norm_col in filtered_df:
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df[norm_col].dropna(), bins=20, edgecolor='k', alpha=0.7)
    plt.xlabel(f"Normalized {numeric_field_id}")
    plt.ylabel("Frequency")
    plt.title(f"Distribution of normalized '{numeric_field_id}' in filtered records")
    plt.grid()
    plt.show()

    # Optional: Bar plot by group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df.plot.bar(x=group_field_id, y=numeric_field_id, figsize=(8,5), legend=False)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Run the EDA section above first to select fields for visualization.")

## 6. Conclusion
This notebook demonstrated how to explore the FAIR² mass spectrometry dataset, including:
- Loading dataset metadata and record sets using `mlcroissant`
- Inspecting available record sets, fields, and their `@id`s
- Extracting records into Pandas DataFrames
- Performing basic filtering, normalization, and grouping
- Visualizing distributions and grouped statistics

**You can further extend this workflow by experimenting with advanced data analysis and adding your own biological or proteomic domain knowledge for interpretation.**
